In [1]:
import pandas as pd

print("Notebook working successfully")

Notebook working successfully


In [4]:
import pandas as pd

df = pd.read_csv("../data/raw/accepted_2007_to_2018Q4.csv")

df.head()
df.shape
df.columns.tolist()
df['loan_status'].value_counts()


MemoryError: Unable to allocate 32.0 KiB for an array with shape (4096,) and data type float64

In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/accepted_2007_to_2018Q4.csv",
    nrows=50000,
    low_memory=False
)

print(df.shape)

(50000, 151)


In [2]:
df.columns.tolist()

['id',
 'member_id',
 'loan_amnt',
 'funded_amnt',
 'funded_amnt_inv',
 'term',
 'int_rate',
 'installment',
 'grade',
 'sub_grade',
 'emp_title',
 'emp_length',
 'home_ownership',
 'annual_inc',
 'verification_status',
 'issue_d',
 'loan_status',
 'pymnt_plan',
 'url',
 'desc',
 'purpose',
 'title',
 'zip_code',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'earliest_cr_line',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'mths_since_last_delinq',
 'mths_since_last_record',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'initial_list_status',
 'out_prncp',
 'out_prncp_inv',
 'total_pymnt',
 'total_pymnt_inv',
 'total_rec_prncp',
 'total_rec_int',
 'total_rec_late_fee',
 'recoveries',
 'collection_recovery_fee',
 'last_pymnt_d',
 'last_pymnt_amnt',
 'next_pymnt_d',
 'last_credit_pull_d',
 'last_fico_range_high',
 'last_fico_range_low',
 'collections_12_mths_ex_med',
 'mths_since_last_major_derog',
 'policy_code',
 'application_type',
 'annual_inc_joint',
 '

In [3]:
df["loan_status"].value_counts()

loan_status
Fully Paid            34978
Charged Off            9027
Current                5610
Late (31-120 days)      246
In Grace Period         100
Late (16-30 days)        38
Default                   1
Name: count, dtype: int64

In [4]:
df_model = df[
    df["loan_status"].isin(
        ["Fully Paid", "Charged Off", "Default"]
    )
].copy()

df_model.shape

(44006, 151)

In [5]:
df_model["default"] = df_model["loan_status"].apply(
    lambda x: 1 if x in ["Charged Off", "Default"] else 0
)

df_model["default"].value_counts()

default
0    34978
1     9028
Name: count, dtype: int64

In [6]:
df_model["default"].mean()

np.float64(0.20515384265781939)

In [7]:
df_model.groupby("default")["annual_inc"].mean()

default
0    80100.016092
1    72211.673939
Name: annual_inc, dtype: float64

In [8]:
df_model.groupby("default")["loan_amnt"].mean()

default
0    13993.264338
1    15612.801839
Name: loan_amnt, dtype: float64

In [9]:
missing = (
    df_model.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing.head(20)

member_id                                     44006
sec_app_collections_12_mths_ex_med            44006
sec_app_chargeoff_within_12_mths              44006
sec_app_num_rev_accts                         44006
sec_app_open_act_il                           44006
sec_app_revol_util                            44006
sec_app_open_acc                              44006
sec_app_mort_acc                              44006
sec_app_mths_since_last_major_derog           44006
sec_app_earliest_cr_line                      44006
sec_app_fico_range_high                       44006
sec_app_inq_last_6mths                        44006
sec_app_fico_range_low                        44006
revol_bal_joint                               44006
next_pymnt_d                                  44005
desc                                          44003
verification_status_joint                     43824
dti_joint                                     43824
annual_inc_joint                              43824
orig_project

In [10]:
features = [
    "annual_inc",
    "loan_amnt",
    "dti",
    "int_rate",
    "installment"
]

df_model.groupby("default")[features].mean()

,annual_inc,loan_amnt,dti,int_rate,installment
default,,,,,
0,80100.016092,13993.264338,18.546077,11.330171,423.169508
1,72211.673939,15612.801839,21.171202,14.560410,450.951800


In [11]:
pd.crosstab(
    df_model["home_ownership"],
    df_model["default"],
    normalize="index"
) * 100

default,0,1
home_ownership,,
ANY,100.000000,0.000000
MORTGAGE,82.386870,17.613130
OWN,78.797964,21.202036
RENT,76.160938,23.839062


In [12]:
pd.crosstab(
    df_model["purpose"],
    df_model["default"],
    normalize="index"
) * 100

default,0,1
purpose,,
car,84.500000,15.500000
credit_card,82.981434,17.018566
debt_consolidation,77.685148,22.314852
home_improvement,82.122695,17.877305
house,72.781065,27.218935
major_purchase,79.791667,20.208333
medical,77.231330,22.768670
moving,76.226415,23.773585
other,80.893204,19.106796


In [13]:
df_model["emp_length"].value_counts()

emp_length
10+ years    14518
< 1 year      3990
2 years       3905
3 years       3574
1 year        2906
5 years       2686
4 years       2523
8 years       2186
6 years       1739
9 years       1658
7 years       1564
Name: count, dtype: int64

In [14]:
important_cols = [
    "loan_status",
    "loan_amnt",
    "annual_inc",
    "dti",
    "int_rate",
    "installment",
    "home_ownership",
    "purpose",
    "emp_length",
    "grade",
    "sub_grade",
    "fico_range_low",
    "fico_range_high"
]

df_model[important_cols].info()

<class 'pandas.DataFrame'>
Index: 44006 entries, 0 to 49999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   loan_status      44006 non-null  str    
 1   loan_amnt        44006 non-null  float64
 2   annual_inc       44006 non-null  float64
 3   dti              44005 non-null  float64
 4   int_rate         44006 non-null  float64
 5   installment      44006 non-null  float64
 6   home_ownership   44006 non-null  str    
 7   purpose          44006 non-null  str    
 8   emp_length       41249 non-null  str    
 9   grade            44006 non-null  str    
 10  sub_grade        44006 non-null  str    
 11  fico_range_low   44006 non-null  float64
 12  fico_range_high  44006 non-null  float64
dtypes: float64(7), str(6)
memory usage: 6.5 MB


In [15]:
df_model[important_cols].head()

,loan_status,loan_amnt,annual_inc,dti,int_rate,installment,home_ownership,purpose,emp_length,grade,sub_grade,fico_range_low,fico_range_high
0,Fully Paid,3600.0,55000.0,5.91,13.99,123.03,MORTGAGE,debt_consolidation,10+ years,C,C4,675.0,679.0
1,Fully Paid,24700.0,65000.0,16.06,11.99,820.28,MORTGAGE,small_business,10+ years,C,C1,715.0,719.0
2,Fully Paid,20000.0,63000.0,10.78,10.78,432.66,MORTGAGE,home_improvement,10+ years,B,B4,695.0,699.0
4,Fully Paid,10400.0,104433.0,25.37,22.45,289.91,MORTGAGE,major_purchase,3 years,F,F1,695.0,699.0
5,Fully Paid,11950.0,34000.0,10.20,13.44,405.18,RENT,debt_consolidation,4 years,C,C3,690.0,694.0
